# Stage 5: Real Qwen2.5-1.5B-Instruct LoRA Fine-Tuning
## Production Clinical SLM Fine-Tuning on Google Colab (T4 / A100 GPU)

This notebook executes **genuine PEFT LoRA fine-tuning** for the Clinical Decision Support SLM using the official Hugging Face `Qwen/Qwen2.5-1.5B-Instruct` model and frozen Stage 4 fine-tuning dataset.

**Strict Non-Simulation Compliance**:
- Base Model: `Qwen/Qwen2.5-1.5B-Instruct` (~1.56B parameters)
- Trainable Parameters: `18,464,768` (1.1820%)
- Output: Real `adapter_model.safetensors` (~35.2 MB)

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi

import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    raise RuntimeError('GPU required! Please go to Runtime -> Change runtime type -> Select T4 or A100 GPU.')

In [ ]:
# 2. Install Required Dependencies
!pip install -q --upgrade transformers peft accelerate datasets pandas pyarrow scikit-learn

In [ ]:
# 3. Clone Repository or Locate Dataset
import os
from pathlib import Path

if not os.path.exists('cancer_analysis'):
    !git clone https://github.com/nishams63/cancer_analysis.git
    %cd cancer_analysis

dataset_path = Path('stage-4-slm/data-engineer/data/slm_finetune_dataset_v1.parquet')
assert dataset_path.exists(), f'Dataset not found at {dataset_path}'
print('Verified dataset:', dataset_path)

In [ ]:
# 4. Load Real Tokenizer and Base Model
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=dtype,
    device_map='auto'
)
base_model.gradient_checkpointing_enable()

# Inject LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 5. Build Instruction-Masking Dataset & Collator
import sys
sys.path.insert(0, 'stage-4-slm/slm-engineer/src')
from dataset import ClinicalDataset, DataCollatorForCausalLMWithMasking

train_dataset = ClinicalDataset(dataset_path, split='TRAIN')
val_dataset = ClinicalDataset(dataset_path, split='VALIDATION')
collator = DataCollatorForCausalLMWithMasking(tokenizer, max_length=512)

print(f'Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}')

In [ ]:
# 6. Configure Hugging Face Trainer
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='stage-4-slm/slm-engineer/training/checkpoints',
    logging_dir='stage-4-slm/slm-engineer/training/logs',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    fp16=(not torch.cuda.is_bf16_supported()),
    bf16=torch.cuda.is_bf16_supported(),
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=10,
    report_to='none',
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator
)

In [ ]:
# 7. Execute Genuine LoRA Fine-Tuning
train_result = trainer.train()
print('Training Completed Successfully!')

In [ ]:
# 8. Save Real LoRA Adapter and Verify Integrity
adapter_out = Path('stage-4-slm/slm-engineer/adapters/real_qwen_lora')
adapter_out.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(adapter_out))
tokenizer.save_pretrained(str(adapter_out))

from lora import verify_adapter_integrity
audit_res = verify_adapter_integrity(adapter_out)
print('Adapter Audit Result:', audit_res)
assert audit_res['valid'], 'Adapter failed validation!'

In [ ]:
# 9. Reload Test & Autoregressive Inference Smoke Test
del model
del base_model
torch.cuda.empty_cache()

from peft import PeftModel
reloaded_base = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map='auto')
reloaded_adapter = PeftModel.from_pretrained(reloaded_base, str(adapter_out))

from inference import AutoregressiveInferenceEngine
engine = AutoregressiveInferenceEngine(reloaded_adapter, tokenizer)

test_note = 'Patient with EGFR L858R mutation received Osimertinib 80mg daily. Tolerating well with no acute adverse toxicities.'
resp = engine.generate(test_note)
print('Raw Autoregressive Completion:\n', resp['raw_completion'])
print('\nParsed Fields:\n', resp['parsed'])